In [7]:
import pandas as pd

df = pd.read_csv('dados_brutos/faixa_etaria_ibge.csv')
df

,Codigo,Municipio,Idade,Total
0,1500107,Abaetetuba (PA),Total,141100.0
1,1500107,Abaetetuba (PA),Menos de 1 ano,2581.0
2,1500107,Abaetetuba (PA),3 anos,2675.0
3,1500107,Abaetetuba (PA),4 anos,2643.0
4,1500107,Abaetetuba (PA),6 anos,2666.0
...,...,...,...,...
3441,0,Zero resultante de um cálculo ou arredondament...,NaN,NaN
3442,X,Valor inibido para não identificar o informant...,NaN,NaN
3443,..,Valor não se aplica.\nEx: Não se pode obter o ...,NaN,NaN
3444,...,Valor não disponível.\nEx: A produção de feijã...,NaN,NaN


In [8]:


df = df[
    pd.to_numeric(df['Codigo'], errors='coerce').notna()
].copy()

df['Codigo'] = df['Codigo'].astype(int)

# Remove o último dígito do código IBGE
df['Codigo'] = df['Codigo'] // 10


# --------------------------------
# 2. Extrair idade numérica
# --------------------------------

def extrair_idade(x):
    if x == 'Menos de 1 ano':
        return 0
    
    if isinstance(x, str) and 'anos' in x:
        return int(x.split()[0])
    
    return None


df['idade_num'] = df['Idade'].apply(extrair_idade)


# --------------------------------
# 3. Criar faixas etárias
# --------------------------------

def faixa_idade(idade):
    if pd.isna(idade):
        return None
    elif idade <= 3:
        return '0-3'
    elif idade <= 6:
        return '4-6'
    elif idade <= 15:
        return '7-15'
    elif idade <= 17:
        return '16-17'
    elif idade <= 24:
        return '18-24'
    elif idade <= 34:
        return '25-34'
    elif idade <= 39:
        return '35-39'
    elif idade <= 44:
        return '40-44'
    elif idade <= 49:
        return '45-49'
    elif idade <= 54:
        return '50-54'
    elif idade <= 59:
        return '55-59'
    elif idade <= 64:
        return '60-64'
    else:
        return '64+'


df['Faixa'] = df['idade_num'].apply(faixa_idade)


# --------------------------------
# 4. Total por faixa
# --------------------------------

df_faixas = (
    df[df['Faixa'].notna()]
    .groupby(
        ['Codigo', 'Municipio', 'Faixa'],
        as_index=False
    )['Total']
    .sum()
)


# --------------------------------
# 5. Transformar faixas em colunas
# --------------------------------

df_final = df_faixas.pivot(
    index=['Codigo', 'Municipio'],
    columns='Faixa',
    values='Total'
).reset_index()


# --------------------------------
# 6. Garantir todas as faixas
# --------------------------------

faixas = [
    '0-3',
    '4-6',
    '7-15',
    '16-17',
    '18-24',
    '25-34',
    '35-39',
    '40-44',
    '45-49',
    '50-54',
    '55-59',
    '60-64',
    '64+'
]

for faixa in faixas:
    if faixa not in df_final.columns:
        df_final[faixa] = pd.NA


# --------------------------------
# 7. Recuperar Total original
# --------------------------------

df_total = df[df['Idade'] == 'Total'][
    ['Codigo', 'Municipio', 'Total']
].copy()

df_final = df_final.merge(
    df_total,
    on=['Codigo', 'Municipio'],
    how='left'
)


# --------------------------------
# 8. Município
# --------------------------------

df_final['Municipio'] = (
    df_final['Municipio']
    .str.replace(r' \(PA\)$', '', regex=True)
    .str.upper()
)


# --------------------------------
# 9. Ordem final
# --------------------------------

df_final = df_final[
    ['Codigo', 'Municipio'] + faixas + ['Total']
]

In [9]:
df_final.to_csv('dados_tratados/faixa_etaria_ibge.csv',index=False)